In [1]:
%matplotlib tk
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from matplotlib import pyplot as plt

In [2]:
url = 'https://github.com/CSSEGISandData/COVID-19/raw/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv'
url = 'https://github.com/CSSEGISandData/COVID-19/raw/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv'
df = pd.read_csv(url,index_col=1,parse_dates=[0])
df = df.T

In [3]:
time_series = df['South Africa'].drop(['Province/State', 'Lat', 'Long'])
last_date = time_series.keys()[-1]
last_date
first_date = time_series.keys()[0]
first_date

'1/22/20'

First death is on 27 March.

First confirmed case on 5 March

In [4]:
first_date = '3/27/20'
first_date = '3/5/20'
# first_date = '10/21/20'  # second wave

In [5]:
total_deaths = time_series.values[time_series.values.nonzero()]
# total_deaths = total_deaths[230:]  # only second wave

In [6]:
deaths_day = np.diff(total_deaths)
plt.plot(deaths_day)
plt.show()

In [7]:
# gamma = 3
# q = 1.26


def fitfunc(t, C, alpha, beta, gamma, q):
    gamma = 3
    q = 1.26
    num = C * t ** alpha
    expo = 1 / (q - 1)
    denom = (1 + (q - 1) * beta * t ** gamma) ** expo
    return num / denom


t = np.arange(np.size(deaths_day))
param, pcov = curve_fit(fitfunc, t, deaths_day, p0=[1e-4, 5, 1e-6, 3, 1.26], bounds=(0, np.inf))
C = param[0]
alpha = param[1]
beta = param[2]
gamma = param[3]
q = param[4]
print(param)
print(pcov)
std_C = np.sqrt(pcov[0, 0])
std_alpha = np.sqrt(pcov[1, 1])
std_beta = np.sqrt(pcov[2, 2])

# alpha = 5.58
# beta = 4e-7
# C = 80


fitted = fitfunc(t, C, alpha, beta, gamma, q)
# fitted_low = fitfunc(t, C - std_C, alpha - std_alpha, beta + std_beta)
# fitted_high = fitfunc(t, C + std_C, alpha + std_alpha, beta - std_beta)
plt.plot(t, deaths_day, 'o')
plt.plot(t, fitted)
# plt.plot(t, fitted_low)
# plt.plot(t, fitted_high)
plt.show()

[4.50036519e-05 4.17911542e+00 7.85584030e-07 3.00000000e+00
 1.26000000e+00]
[[ 1.45702810e-07 -7.44959228e-04 -2.46497224e-10  0.00000000e+00
   0.00000000e+00]
 [-7.44959228e-04  3.81068721e+00  1.26438738e-06  0.00000000e+00
   0.00000000e+00]
 [-2.46497224e-10  1.26438738e-06  4.27308912e-13  0.00000000e+00
   0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]]


In [8]:
t_future = np.arange(400)
extrap = fitfunc(t_future, C, alpha, beta, gamma, q)
plt.plot(t, deaths_day, 'o')
plt.plot(t_future, extrap)
plt.show()

In [9]:
extrap_series = pd.Series(extrap, index=pd.date_range(first_date, freq='D', periods=len(extrap)))

In [10]:
extrap_series.plot()

<AxesSubplot:>

In [11]:
plt.plot(np.cumsum(extrap_series.values))

In [12]:
deaths_day = np.pad(deaths_day, (0, 200))
extrap_series = pd.Series(deaths_day, index=pd.date_range(first_date, freq='D', periods=len(deaths_day)))

In [13]:

extrap_series.plot()

<AxesSubplot:>